# Naïve vs ML Forecasting
This companion notebook runs **offline** with synthetic OHLCV data. Use the Streamlit dashboard for a complete interactive interface.

Run Jupyter from the project root or the `ml/` folder. Predictions are illustrative and not trading advice.

In [ ]:
import sys
from pathlib import Path

# Resolve the package root whether Jupyter starts at root or ml/.
for folder in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (folder / "ml" / "src" / "comparison.py").is_file():
        if str(folder) not in sys.path:
            sys.path.insert(0, str(folder))
        break
else:
    raise FileNotFoundError("Start Jupyter within the extracted project directory")

import pandas as pd
import plotly.graph_objects as go
from ml.src.sample_data import make_demo_data
from ml.src.comparison import run_comparison
from ml.src.data_fetch import get_data

## 1. Offline benchmark
The naïve prediction repeats the most recent close for all horizons. The Random Forest is trained using earlier windows only and compared on a chronological holdout.

In [ ]:
prices = make_demo_data(rows=900, seed=23)
report = run_comparison(prices, ticker="DEMO", lookback=30, horizon=7,
                        ml_model="random_forest", n_estimators=100,
                        calendar_mode="weekdays")
summary = pd.DataFrame({
    "Naïve": {k:v for k,v in report.metrics["naive"].items() if k != "per_horizon"},
    "Random Forest": {k:v for k,v in report.metrics["ml"].items() if k != "per_horizon"}
})
display(summary)
display(report.forecast)

## 2. Inspect future forecasts and held-out predictions

In [ ]:
fig = go.Figure()
recent = report.historical.tail(90)
fig.add_scatter(x=recent.index, y=recent.Close, name="Observed")
fig.add_scatter(x=report.forecast.Forecast_Date, y=report.forecast.Naive_Close, name="Naïve")
fig.add_scatter(x=report.forecast.Forecast_Date, y=report.forecast.ML_Close, name="Random Forest")
fig.update_layout(title="Synthetic demo — future forecasts", xaxis_title="Date", yaxis_title="Close")
fig.show()

In [ ]:
fig = go.Figure()
for column,name in [("Actual_Close","Actual"),("Naive_Close","Naïve"),("ML_Close","ML")]:
    fig.add_scatter(x=report.backtest.Date, y=report.backtest[column], name=name)
fig.update_layout(title="Held-out test: next-session forecasts", xaxis_title="Date")
fig.show()

## 3. Live data or Transformer
To use live Yahoo Finance prices, run:
```python
prices = get_data("AAPL", "5y", "1d")
report = run_comparison(prices, ticker="AAPL", exchange="NYSE", ml_model="random_forest")
```
For the Transformer, change `ml_model="transformer"` and optionally `epochs=12`. Live data requires an internet connection and the `pandas-market-calendars` package. Historical benchmarks are **not** trading signals.